# ENARES 2024 CRS04 — Stage 03
## Notebook 11 · Reproducibility Check / Issue #40

Este notebook **no transforma ni recrea datos**. Su función es verificar que Stage 03
pueda reconstruirse de manera auditable desde los insumos de Stage 02, los notebooks/SQL
versionados y los scripts R.

### Evidencia producida

- `05Resultados/logs/stage03/stage3_reproducibility_check.csv`
- `05Resultados/logs/stage03/stage3_reproducibility_hashes.csv`
- `05Resultados/logs/stage03/stage3_reproducibility.md`

### Regla de decisión

`REPRODUCIBILITY PASS` exige que estén disponibles los insumos, tablas, diccionario,
indicadores, SQL/scripts y productos R necesarios para reconstruir Stage 03.

La comparación `SPSS vs R` se reporta por separado porque pertenece al cierre estadístico
de Stage 03. Si falta, **no invalida la capacidad técnica de reconstrucción**, pero sí
mantiene pendiente el cierre completo / `stage3_pass.md`.

In [1]:
!pip install -q google-cloud-bigquery pandas pyarrow db-dtypes tabulate

In [2]:
from google.colab import auth, drive
from google.cloud import bigquery
from google.api_core.exceptions import NotFound
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import hashlib
import json
import os

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = "enares-2024-crs04"
LOCATION = "US"
EXPECTED_ROWS = 18807

ROOT = Path("/content/drive/MyDrive/ENARES_2024_PROJECT")
LOG_DIR = ROOT / "05Resultados" / "logs" / "stage03"
SQL_DIR = ROOT / "02SQL"
OUTPUT_DIR = ROOT / "04Outputs"
R_DIR = ROOT / "03Scripts_R"
DOCS_DIR = ROOT / "docs"

for d in [LOG_DIR, SQL_DIR, OUTPUT_DIR, R_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

RAW_DS = f"{PROJECT_ID}.enares2024_crs04_raw"
CLEANED = f"{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents"
ANALYTICAL = f"{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents"

print("RUN_UTC:", RUN_UTC)
print("Proyecto:", PROJECT_ID)

Mounted at /content/drive
RUN_UTC: 2026-08-12T01:06:29.307078+00:00
Proyecto: enares-2024-crs04


## 1. Orden canónico de reconstrucción

Este orden evita el problema de ejecutar la tabla analytical base después de los módulos
derivados y borrar columnas ya materializadas.

In [3]:
reconstruction_order = pd.DataFrame([
    (1, "Stage 02", "Raw + metadata", "raw_crs04_cap100/200/248/300 + metadata_*"),
    (2, "01", "Setup / validation", "prerequisites, key, FLOAT64, collisions"),
    (3, "08", "Analytical base FINAL", "crear/recrear analytical base"),
    (4, "02", "Violencia hogar", "módulo hogar"),
    (5, "03", "Violencia escuela", "VP_ESCUELA, VF_ESCUELA, INDICADOR_8_3_9, etc."),
    (6, "04", "Violencia sexual COMPLETO", "INDICADOR_8_3_6 + VS_12M"),
    (7, "05", "Acumulación de violencias", "PV_*"),
    (8, "06", "Consecuencias", "CONS_*"),
    (9, "07", "Búsqueda de ayuda", "ayuda_*, DEMUNA, brechas"),
    (10, "04R", "Riesgo/desprotección CRS04", "rd12_*"),
    (11, "10", "Diccionario + linaje", "diccionario_indicadores + stage3_lineage"),
    (12, "R1", "Export / survey design", "analytical_for_r.csv + design_crs04.rds"),
    (13, "R2", "Tabulados finales", "tabulados_crs04_long.csv"),
    (14, "09", "Validación y cierre", "SPSS vs R + closure/pass"),
    (15, "11", "Reproducibilidad", "este notebook"),
], columns=["step", "component", "action", "expected_output"])

display(reconstruction_order)

reconstruction_order.to_csv(
    LOG_DIR / "stage3_reconstruction_order.csv",
    index=False
)

,step,component,action,expected_output
0,1,Stage 02,Raw + metadata,raw_crs04_cap100/200/248/300 + metadata_*
1,2,01,Setup / validation,"prerequisites, key, FLOAT64, collisions"
2,3,08,Analytical base FINAL,crear/recrear analytical base
3,4,02,Violencia hogar,módulo hogar
4,5,03,Violencia escuela,"VP_ESCUELA, VF_ESCUELA, INDICADOR_8_3_9, etc."
5,6,04,Violencia sexual COMPLETO,INDICADOR_8_3_6 + VS_12M
6,7,05,Acumulación de violencias,PV_*
7,8,06,Consecuencias,CONS_*
8,9,07,Búsqueda de ayuda,"ayuda_*, DEMUNA, brechas"
9,10,04R,Riesgo/desprotección CRS04,rd12_*


## 2. Verificar insumos de Stage 02 en BigQuery

In [4]:
required_raw_tables = [
    "raw_crs04_cap100",
    "raw_crs04_cap200",
    "raw_crs04_cap248",
    "raw_crs04_cap300",
    "metadata_crs04_variables",
    "metadata_crs04_value_labels",
    "metadata_crs04_missing_codes",
    "metadata_crs04_source_files",
]

bq_checks = []

for name in required_raw_tables:
    table_id = f"{RAW_DS}.{name}"
    try:
        t = client.get_table(table_id)
        bq_checks.append({
            "category": "stage02_bigquery",
            "item": name,
            "required": True,
            "exists": True,
            "rows": int(t.num_rows),
            "detail": table_id,
        })
    except NotFound:
        bq_checks.append({
            "category": "stage02_bigquery",
            "item": name,
            "required": True,
            "exists": False,
            "rows": None,
            "detail": table_id,
        })

stage02_bq = pd.DataFrame(bq_checks)
display(stage02_bq)

if not stage02_bq["exists"].all():
    missing = stage02_bq.loc[~stage02_bq["exists"], "item"].tolist()
    print("FALTAN insumos Stage 02:", missing)
else:
    print("PASS: raw + metadata Stage 02 disponibles.")

,category,item,required,exists,rows,detail
0,stage02_bigquery,raw_crs04_cap100,True,True,18807,enares-2024-crs04.enares2024_crs04_raw.raw_crs...
1,stage02_bigquery,raw_crs04_cap200,True,True,18807,enares-2024-crs04.enares2024_crs04_raw.raw_crs...
2,stage02_bigquery,raw_crs04_cap248,True,True,18807,enares-2024-crs04.enares2024_crs04_raw.raw_crs...
3,stage02_bigquery,raw_crs04_cap300,True,True,18807,enares-2024-crs04.enares2024_crs04_raw.raw_crs...
4,stage02_bigquery,metadata_crs04_variables,True,True,1299,enares-2024-crs04.enares2024_crs04_raw.metadat...
5,stage02_bigquery,metadata_crs04_value_labels,True,True,3317,enares-2024-crs04.enares2024_crs04_raw.metadat...
6,stage02_bigquery,metadata_crs04_missing_codes,True,True,0,enares-2024-crs04.enares2024_crs04_raw.metadat...
7,stage02_bigquery,metadata_crs04_source_files,True,True,4,enares-2024-crs04.enares2024_crs04_raw.metadat...


PASS: raw + metadata Stage 02 disponibles.


## 3. Verificar capas cleaned y analytical

Ambas deben conservar el universo CRS04 de **18,807** adolescentes.

In [5]:
layer_rows = []

for layer, table_id in [
    ("cleaned", CLEANED),
    ("analytical", ANALYTICAL),
]:
    try:
        t = client.get_table(table_id)
        layer_rows.append({
            "category": "bigquery_layer",
            "item": layer,
            "required": True,
            "exists": True,
            "rows": int(t.num_rows),
            "expected_rows": EXPECTED_ROWS,
            "rowcount_ok": int(t.num_rows) == EXPECTED_ROWS,
            "columns": len(t.schema),
            "detail": table_id,
        })
    except NotFound:
        layer_rows.append({
            "category": "bigquery_layer",
            "item": layer,
            "required": True,
            "exists": False,
            "rows": None,
            "expected_rows": EXPECTED_ROWS,
            "rowcount_ok": False,
            "columns": None,
            "detail": table_id,
        })

layers = pd.DataFrame(layer_rows)
display(layers)

if not layers["rowcount_ok"].all():
    print("FAIL: cleaned/analytical no cumplen el universo esperado.")
else:
    print("PASS: cleaned y analytical conservan 18,807 filas.")

,category,item,required,exists,rows,expected_rows,rowcount_ok,columns,detail
0,bigquery_layer,cleaned,True,True,18807,18807,True,1206,enares-2024-crs04.enares2024_crs04_cleaned.cle...
1,bigquery_layer,analytical,True,True,18807,18807,True,1407,enares-2024-crs04.enares2024_crs04_analytical....


PASS: cleaned y analytical conservan 18,807 filas.


## 4. Diccionario vs schema analytical

Se usa el diccionario generado por Stage 03 como contrato ejecutable. No se codifica
manualmente una lista de 78 variables: el notebook lee el artefacto real para evitar
schema drift.

In [6]:
dictionary_candidates = [
    OUTPUT_DIR / "diccionario_indicadores.csv",
    LOG_DIR / "diccionario_indicadores.csv",
]

dictionary_path = next((p for p in dictionary_candidates if p.exists()), None)

if dictionary_path is None:
    raise FileNotFoundError(
        "No se encontró diccionario_indicadores.csv. Ejecuta primero el notebook 10."
    )

dic = pd.read_csv(dictionary_path)

if "indicator_name" not in dic.columns:
    raise RuntimeError(
        f"El diccionario {dictionary_path.name} no contiene indicator_name."
    )

if "status" in dic.columns:
    implemented = dic[
        dic["status"].astype(str).str.lower().eq("implemented")
    ].copy()
else:
    implemented = dic.copy()

implemented_names = (
    implemented["indicator_name"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

analytical_table = client.get_table(ANALYTICAL)
analytical_columns = {f.name for f in analytical_table.schema}

indicator_schema_check = pd.DataFrame({
    "indicator_name": implemented_names
})
indicator_schema_check["exists_in_analytical"] = (
    indicator_schema_check["indicator_name"].isin(analytical_columns)
)

display(indicator_schema_check)

print("Indicadores implementados en diccionario:", len(indicator_schema_check))
print("Encontrados:", int(indicator_schema_check["exists_in_analytical"].sum()))
print("Faltantes:", int((~indicator_schema_check["exists_in_analytical"]).sum()))

missing_indicators = indicator_schema_check[
    ~indicator_schema_check["exists_in_analytical"]
]

if len(missing_indicators):
    display(missing_indicators)
else:
    print("PASS: todos los indicadores implementados existen en analytical.")

,indicator_name,exists_in_analytical
0,cree_al_menos_un_mito,True
1,indice_derechos,True
2,justifica_al_menos_una,True
3,justifica_castigo_docente,True
4,justifica_castigo_parental,True
...,...,...
73,recibio_ayuda_vs,True
74,recibio_ayuda_vs_victimas,True
75,uso_demuna,True
76,rd12_idx_noviol,True


Indicadores implementados en diccionario: 78
Encontrados: 78
Faltantes: 0
PASS: todos los indicadores implementados existen en analytical.


## 5. Validación de dominio desde el diccionario

Solo se validan como binarios los indicadores cuyo contrato declara valores
`0,1`, `0,1,NULL` o equivalente. Índices/conteos/categorías no se fuerzan a binario.

In [7]:
valid_col_candidates = [
    "valid_values",
    "domain",
    "values",
    "valid_domain",
]

valid_col = next(
    (c for c in valid_col_candidates if c in implemented.columns),
    None
)

binary_names = []

if valid_col is not None:
    for _, row in implemented.iterrows():
        name = str(row["indicator_name"])
        domain = str(row[valid_col]).replace(" ", "").upper()
        allowed_binary_domains = {
            "0,1",
            "0,1,NULL",
            "0/1",
            "0/1/NULL",
            "{0,1}",
            "{0,1,NULL}",
        }
        if domain in allowed_binary_domains:
            binary_names.append(name)

# Respaldo por indicator_type solo cuando el diccionario lo declara explícitamente.
if "indicator_type" in implemented.columns:
    typed_binary = implemented.loc[
        implemented["indicator_type"]
        .astype(str)
        .str.lower()
        .str.contains("binary", na=False),
        "indicator_name"
    ].astype(str).tolist()
    binary_names = sorted(set(binary_names) | set(typed_binary))

binary_names = [x for x in binary_names if x in analytical_columns]

domain_rows = []

for name in binary_names:
    q = f'''
    SELECT
      COUNT(*) AS total_rows,
      COUNTIF(`{name}` IS NOT NULL AND `{name}` NOT IN (0,1)) AS invalid_values
    FROM `{ANALYTICAL}`
    '''
    res = client.query(q, location=LOCATION).result().to_dataframe().iloc[0]
    domain_rows.append({
        "indicator_name": name,
        "total_rows": int(res["total_rows"]),
        "invalid_values": int(res["invalid_values"]),
        "domain_ok": int(res["invalid_values"]) == 0,
    })

binary_domain_check = pd.DataFrame(domain_rows)

if len(binary_domain_check):
    display(binary_domain_check)
    print(
        "Binary domain PASS:",
        bool(binary_domain_check["domain_ok"].all())
    )
else:
    print(
        "INFO: el diccionario no expuso una columna de dominio/tipo "
        "suficiente para identificar indicadores binarios automáticamente."
    )

INFO: el diccionario no expuso una columna de dominio/tipo suficiente para identificar indicadores binarios automáticamente.


## 6. Artefactos requeridos en Drive

Se aceptan nombres históricos y nombres finales cuando hubo refactorización.
El check registra exactamente qué ruta encontró.

In [8]:
def first_existing(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None

artifact_specs = [
    ("lineage", True, [
        LOG_DIR / "stage3_lineage.csv",
    ]),
    ("r_export_manifest", True, [
        LOG_DIR / "stage3_r_export_manifest.csv",
    ]),
    ("analytical_for_r", True, [
        OUTPUT_DIR / "analytical_crs04_adolescents_for_r.csv",
    ]),
    ("survey_design_r", True, [
        R_DIR / "survey_design.R",
    ]),
    ("tabulados_r", True, [
        R_DIR / "tabulados.R",
        R_DIR / "tabulados.R",
    ]),
    ("design_rds", True, [
        OUTPUT_DIR / "design_crs04.rds",
    ]),
    ("tabulados_long", True, [
        OUTPUT_DIR / "tabulados_crs04_long.csv",
    ]),
    ("indicator_dictionary_csv", True, [
        OUTPUT_DIR / "diccionario_indicadores.csv",
        LOG_DIR / "diccionario_indicadores.csv",
    ]),
    ("indicator_dictionary_xlsx", False, [
        OUTPUT_DIR / "diccionario_indicadores.xlsx",
    ]),
    ("stage3_description", True, [
        DOCS_DIR / "stage3_description.md",
    ]),
    ("stage3_data_dictionary_md", True, [
        DOCS_DIR / "stage3_data_dictionary.md",
    ]),
    ("spss_vs_r_comparison", False, [
        LOG_DIR / "stage3_spss_vs_r_comparison.csv",
    ]),
    ("spss_reference", False, [
        OUTPUT_DIR / "spss_reference_results.csv",
    ]),
]

artifact_rows = []

for name, required, candidates in artifact_specs:
    found = first_existing(candidates)
    artifact_rows.append({
        "category": "drive_artifact",
        "item": name,
        "required_for_reproducibility": required,
        "exists": found is not None,
        "path": str(found) if found else "",
        "candidates": " | ".join(str(x) for x in candidates),
    })

artifacts = pd.DataFrame(artifact_rows)
display(artifacts)

required_artifacts_ok = artifacts.loc[
    artifacts["required_for_reproducibility"], "exists"
].all()

print("Artefactos obligatorios de reproducibilidad:", bool(required_artifacts_ok))

,category,item,required_for_reproducibility,exists,path,candidates
0,drive_artifact,lineage,True,True,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...
1,drive_artifact,r_export_manifest,True,True,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...
2,drive_artifact,analytical_for_r,True,True,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...
3,drive_artifact,survey_design_r,True,True,/content/drive/MyDrive/ENARES_2024_PROJECT/03S...,/content/drive/MyDrive/ENARES_2024_PROJECT/03S...
4,drive_artifact,tabulados_r,True,True,/content/drive/MyDrive/ENARES_2024_PROJECT/03S...,/content/drive/MyDrive/ENARES_2024_PROJECT/03S...
5,drive_artifact,design_rds,True,True,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...
6,drive_artifact,tabulados_long,True,True,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...
7,drive_artifact,indicator_dictionary_csv,True,True,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...
8,drive_artifact,indicator_dictionary_xlsx,False,True,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...,/content/drive/MyDrive/ENARES_2024_PROJECT/04O...
9,drive_artifact,stage3_description,True,True,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...


Artefactos obligatorios de reproducibilidad: True


## 7. Hashes SHA-256 de código y evidencia

Los hashes permiten demostrar qué versión exacta de SQL/scripts/notebooks produjo el
estado auditado. Solo se hashean archivos que existen.

In [9]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

hash_candidates = []

# SQL producido por Stage 03
if SQL_DIR.exists():
    hash_candidates.extend(sorted(SQL_DIR.glob("stage3*.sql")))

# Scripts R
for p in [
    R_DIR / "survey_design.R",
    R_DIR / "tabulados_crs04_FINAL_desagregados.R",
    R_DIR / "tabulados.R",
]:
    if p.exists():
        hash_candidates.append(p)

# Evidencia de linaje / diccionario / reconstrucción
for p in [
    LOG_DIR / "stage3_lineage.csv",
    OUTPUT_DIR / "diccionario_indicadores.csv",
    LOG_DIR / "stage3_reconstruction_order.csv",
]:
    if p.exists():
        hash_candidates.append(p)

# Notebooks Stage 03 en la raíz del proyecto, si allí están guardados.
for pattern in ["*.ipynb"]:
    for p in ROOT.glob(pattern):
        if (
            "STAGE03" in p.name.upper()
            or "STAGE3" in p.name.upper()
            or p.name.startswith(("01_", "02_", "03_", "04_", "05_", "06_", "07_", "08_", "09_", "10_", "11_"))
        ):
            hash_candidates.append(p)

# Dedupe conservando orden
seen = set()
unique_hash_candidates = []
for p in hash_candidates:
    rp = str(p.resolve())
    if rp not in seen:
        seen.add(rp)
        unique_hash_candidates.append(p)

hash_rows = []
for p in unique_hash_candidates:
    hash_rows.append({
        "file": p.name,
        "path": str(p),
        "size_bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    })

hashes = pd.DataFrame(hash_rows)
hash_path = LOG_DIR / "stage3_reproducibility_hashes.csv"
hashes.to_csv(hash_path, index=False)

display(hashes)
print("Hashes guardados:", hash_path)

,file,path,size_bytes,sha256
0,stage3_32_indicador_836.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,403,627a988e99a21c744701b5275ae23971ecd56edac176a9...
1,stage3_34_indicador_8_3_6.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,686,f87c940a90593e7e72057d84ebc475336cbac770a82230...
2,stage3_34_vs_12m.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1019,f4694ee734d8ea631ce57f07beb456ebbf1fb0dd24aa6e...
3,stage3_35_4_consecuencias.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1525,434ad5f263e648b87cc8907957f14c7ae136d3320eaa51...
4,stage3_35_acumulacion_violencias.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1206,a4ffe2f8e97991ae7d7c3465a611c1b3efd3e3400da188...
5,stage3_36_busqueda_ayuda_vs.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,5045,2a114865b61d0ba2a44bbf7f0c6fe72323c57358019ad3...
6,stage3_create_crs04_analytical.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1261,9e7bc4a9d1be612ea8f0e73a3522dfb3b316856a25ecf5...
7,stage3_create_crs04_cleaned.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,1518,3109ccc786c87cd6e30389bf9f968ecce36ca508d0a034...
8,stage3_ficha_riesgo_desproteccion_crs04.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,16655,28a8e7d26f32af4ec77cb03ca2cdad9a4a3fa40a3690e8...
9,stage3_final_consistency_patch.sql,/content/drive/MyDrive/ENARES_2024_PROJECT/02S...,442,2af2fe6504e3450451de411ee5fa80c730174e0a839d12...


Hashes guardados: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/stage03/stage3_reproducibility_hashes.csv


## 8. Validación de productos R

Se comprueba que el export y los tabulados sean legibles y que el export conserve
18,807 registros.

In [10]:
r_checks = []

export_path = OUTPUT_DIR / "analytical_crs04_adolescents_for_r.csv"
if export_path.exists():
    # Conteo eficiente de filas sin cargar todo el archivo.
    with open(export_path, "rb") as f:
        export_rows = sum(1 for _ in f) - 1
    r_checks.append({
        "item": "analytical_for_r_rows",
        "ok": export_rows == EXPECTED_ROWS,
        "observed": export_rows,
        "expected": EXPECTED_ROWS,
    })
else:
    r_checks.append({
        "item": "analytical_for_r_rows",
        "ok": False,
        "observed": None,
        "expected": EXPECTED_ROWS,
    })

tab_path = OUTPUT_DIR / "tabulados_crs04_long.csv"
if tab_path.exists():
    tab = pd.read_csv(tab_path)
    required_tab_cols = {
        "indicator_id", "dimension", "categoria",
        "pct", "es", "ci_low", "ci_high", "cv", "n_unw"
    }
    r_checks.append({
        "item": "tabulados_long_schema",
        "ok": required_tab_cols.issubset(tab.columns),
        "observed": len(tab.columns),
        "expected": len(required_tab_cols),
    })
    r_checks.append({
        "item": "tabulados_long_nonempty",
        "ok": len(tab) > 0,
        "observed": len(tab),
        "expected": ">0",
    })
else:
    r_checks.extend([
        {
            "item": "tabulados_long_schema",
            "ok": False,
            "observed": None,
            "expected": 9,
        },
        {
            "item": "tabulados_long_nonempty",
            "ok": False,
            "observed": None,
            "expected": ">0",
        },
    ])

r_validation = pd.DataFrame(r_checks)
display(r_validation)

,item,ok,observed,expected
0,analytical_for_r_rows,True,18807,18807
1,tabulados_long_schema,True,9,9
2,tabulados_long_nonempty,True,71,>0


## 9. Decisión reproducibilidad + estado SPSS vs R

`SPSS vs R` se mantiene visible para no confundir **reproducibilidad técnica** con
**cierre estadístico completo**.

In [11]:
checks = []

# Stage 02
for _, row in stage02_bq.iterrows():
    checks.append({
        "check": f"stage02:{row['item']}",
        "required": True,
        "passed": bool(row["exists"]),
        "detail": row["detail"],
    })

# Layers
for _, row in layers.iterrows():
    checks.append({
        "check": f"layer:{row['item']}",
        "required": True,
        "passed": bool(row["exists"]) and bool(row["rowcount_ok"]),
        "detail": f"rows={row['rows']}; expected={EXPECTED_ROWS}",
    })

# Indicators
checks.append({
    "check": "dictionary:all_implemented_indicators_exist",
    "required": True,
    "passed": bool(indicator_schema_check["exists_in_analytical"].all()),
    "detail": (
        f"{int(indicator_schema_check['exists_in_analytical'].sum())}"
        f"/{len(indicator_schema_check)}"
    ),
})

if len(binary_domain_check):
    checks.append({
        "check": "dictionary:binary_domains",
        "required": True,
        "passed": bool(binary_domain_check["domain_ok"].all()),
        "detail": f"{len(binary_domain_check)} binary indicators checked",
    })

# Drive artifacts
for _, row in artifacts.iterrows():
    checks.append({
        "check": f"artifact:{row['item']}",
        "required": bool(row["required_for_reproducibility"]),
        "passed": bool(row["exists"]),
        "detail": row["path"] if row["exists"] else "not found",
    })

# R validation
for _, row in r_validation.iterrows():
    checks.append({
        "check": f"r:{row['item']}",
        "required": True,
        "passed": bool(row["ok"]),
        "detail": f"observed={row['observed']}; expected={row['expected']}",
    })

repro_check = pd.DataFrame(checks)

repro_pass = bool(
    repro_check.loc[
        repro_check["required"], "passed"
    ].all()
)

sp_compare = LOG_DIR / "stage3_spss_vs_r_comparison.csv"
spss_reference = OUTPUT_DIR / "spss_reference_results.csv"

spss_status = (
    "AVAILABLE"
    if sp_compare.exists()
    else "PENDING"
)

repro_check.to_csv(
    LOG_DIR / "stage3_reproducibility_check.csv",
    index=False
)

display(repro_check)

print("=" * 72)
print(
    "REPRODUCIBILITY PASS"
    if repro_pass
    else "REPRODUCIBILITY NOT PASSED"
)
print("SPSS VS R:", spss_status)
print("=" * 72)

,check,required,passed,detail
0,stage02:raw_crs04_cap100,True,True,enares-2024-crs04.enares2024_crs04_raw.raw_crs...
1,stage02:raw_crs04_cap200,True,True,enares-2024-crs04.enares2024_crs04_raw.raw_crs...
2,stage02:raw_crs04_cap248,True,True,enares-2024-crs04.enares2024_crs04_raw.raw_crs...
3,stage02:raw_crs04_cap300,True,True,enares-2024-crs04.enares2024_crs04_raw.raw_crs...
4,stage02:metadata_crs04_variables,True,True,enares-2024-crs04.enares2024_crs04_raw.metadat...
5,stage02:metadata_crs04_value_labels,True,True,enares-2024-crs04.enares2024_crs04_raw.metadat...
6,stage02:metadata_crs04_missing_codes,True,True,enares-2024-crs04.enares2024_crs04_raw.metadat...
7,stage02:metadata_crs04_source_files,True,True,enares-2024-crs04.enares2024_crs04_raw.metadat...
8,layer:cleaned,True,True,rows=18807; expected=18807
9,layer:analytical,True,True,rows=18807; expected=18807


REPRODUCIBILITY PASS
SPSS VS R: PENDING


## 10. Generar reporte Markdown para Issue #40

In [12]:
failed_required = repro_check[
    repro_check["required"] & ~repro_check["passed"]
]["check"].tolist()

optional_missing = repro_check[
    ~repro_check["required"] & ~repro_check["passed"]
]["check"].tolist()

report_lines = [
    "# Stage 03 Reproducibility Check — CRS04",
    "",
    f"- Run UTC: `{RUN_UTC}`",
    f"- Project: `{PROJECT_ID}`",
    f"- Expected analytical universe: `{EXPECTED_ROWS}`",
    f"- Implemented indicators checked: `{len(indicator_schema_check)}`",
    f"- Reproducibility result: `{'PASS' if repro_pass else 'NOT PASSED'}`",
    f"- SPSS vs R status: `{spss_status}`",
    "",
    "## Interpretation",
    "",
    (
        "The technical reconstruction contract is satisfied."
        if repro_pass
        else "The technical reconstruction contract has unresolved required checks."
    ),
    "",
    "SPSS vs R is reported independently because it is a statistical release/closure "
    "criterion. A missing SPSS reference does not prove that the pipeline cannot be "
    "reconstructed, but it prevents claiming full Stage 03 statistical closure.",
    "",
    "## Canonical reconstruction order",
    "",
]

for _, row in reconstruction_order.iterrows():
    report_lines.append(
        f"{int(row['step'])}. **{row['component']}** — "
        f"{row['action']} → `{row['expected_output']}`"
    )

report_lines.extend([
    "",
    "## Required checks not passed",
    "",
])

if failed_required:
    report_lines.extend([f"- `{x}`" for x in failed_required])
else:
    report_lines.append("- None.")

report_lines.extend([
    "",
    "## Optional / closure evidence not available",
    "",
])

if optional_missing:
    report_lines.extend([f"- `{x}`" for x in optional_missing])
else:
    report_lines.append("- None.")

report_lines.extend([
    "",
    "## Evidence files",
    "",
    "- `stage3_reproducibility_check.csv`",
    "- `stage3_reproducibility_hashes.csv`",
    "- `stage3_reconstruction_order.csv`",
    "",
    "## Reproducibility statement",
    "",
    "Stage 03 is considered technically reproducible only when the required checks above "
    "pass using the canonical execution order. BigQuery is treated as a rebuildable "
    "processing layer; Drive preserves inputs/outputs and GitHub should preserve code, "
    "SQL, documentation and non-sensitive logs.",
])

report_path = LOG_DIR / "stage3_reproducibility.md"
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8"
)

print(report_path)
print()
print(report_path.read_text(encoding="utf-8"))

/content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/stage03/stage3_reproducibility.md

# Stage 03 Reproducibility Check — CRS04

- Run UTC: `2026-08-12T01:06:29.307078+00:00`
- Project: `enares-2024-crs04`
- Expected analytical universe: `18807`
- Implemented indicators checked: `78`
- Reproducibility result: `PASS`
- SPSS vs R status: `PENDING`

## Interpretation

The technical reconstruction contract is satisfied.

SPSS vs R is reported independently because it is a statistical release/closure criterion. A missing SPSS reference does not prove that the pipeline cannot be reconstructed, but it prevents claiming full Stage 03 statistical closure.

## Canonical reconstruction order

1. **Stage 02** — Raw + metadata → `raw_crs04_cap100/200/248/300 + metadata_*`
2. **01** — Setup / validation → `prerequisites, key, FLOAT64, collisions`
3. **08** — Analytical base FINAL → `crear/recrear analytical base`
4. **02** — Violencia hogar → `módulo hogar`
5. **03** — Violencia escuela → `

## 11. Assertion final

Esta celda falla solamente si **Issue #40 (reproducibilidad técnica)** no cumple.
No fuerza un falso `PASS` de Stage 03 cuando la comparación SPSS vs R sigue pendiente.

In [13]:
if not repro_pass:
    failed = repro_check.loc[
        repro_check["required"] & ~repro_check["passed"],
        "check"
    ].tolist()
    raise RuntimeError(
        "Issue #40 — reproducibilidad NO aprobada. Faltan/fallan: "
        + ", ".join(failed)
    )

print("PASS Issue #40: Stage 03 es técnicamente reproducible y auditable.")

if spss_status != "AVAILABLE":
    print(
        "PENDIENTE DE CIERRE ESTADÍSTICO: no existe todavía "
        "stage3_spss_vs_r_comparison.csv."
    )

PASS Issue #40: Stage 03 es técnicamente reproducible y auditable.
PENDIENTE DE CIERRE ESTADÍSTICO: no existe todavía stage3_spss_vs_r_comparison.csv.
